# 🏋️ Squat Form Analyzer

Análisis de forma en sentadillas usando **YOLOv8 pose estimation** + **biomecánica**.

Este notebook:
1. Extrae keypoints del cuerpo con YOLOv8-pose
2. Calcula ángulos específicos de sentadilla (rodilla, cadera, espalda)
3. Evalúa la forma usando reglas biomecánicas
4. Guarda el clasificador como modelo joblib

---
## 1. Imports y Configuración

In [ ]:
# === Imports ===
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import joblib
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

# Nuestros módulos
sys.path.insert(0, '.')
from src.pose_extractor import PoseExtractor, KEYPOINT_NAMES
from src.angle_utils import get_squat_angles, aggregate_video_features, ANGLE_EXPLANATIONS
from src.squat_classifier import SquatFormClassifier, CriterionResult

# === Config ===
MODEL_NAME = "yolov8n-pose.pt"  # nano — rápido. Prueba "yolov8s-pose.pt" para más precisión
FRAME_SKIP = 3  # procesar 1 de cada N frames (más rápido)

print("✅ Imports listos")

---
## 2. Cargar Modelo YOLO Pose

In [ ]:
# Cargar el modelo de pose
pose_extractor = PoseExtractor(MODEL_NAME)
print(f"✅ Modelo cargado: {MODEL_NAME}")
print(f"🔑 Keypoints COCO: {len(KEYPOINT_NAMES)}")
for kid, name in KEYPOINT_NAMES.items():
    print(f"   {kid}: {name}")

---
## 3. Probar Extracción en una Imagen

Vamos a probar con la webcam o una imagen de prueba.

In [ ]:
# Función para capturar desde webcam
def capture_webcam(camera_id=0):
    cap = cv2.VideoCapture(camera_id)
    if not cap.isOpened():
        print("❌ No se pudo abrir la cámara")
        return None
    print("📸 Capturando frame...", end=" ")
    ret, frame = cap.read()
    cap.release()
    if ret:
        print("OK!")
        return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    else:
        print("❌ Error al capturar")
        return None

# Intentar con la cámara
frame = capture_webcam()

if frame is None:
    # Crear un frame sintético
    print("ℹ️  No hay cámara — creando frame de demostración")
    frame = np.ones((480, 640, 3), dtype=np.uint8) * 200
    cv2.putText(frame, "Sin camara - usa un video", (100, 240),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2)

print(f"   Frame shape: {frame.shape}")

In [ ]:
# Extraer keypoints
kps, annotated = pose_extractor.extract_from_frame(frame)

print(f"🎯 Keypoints detectados: {len(kps)}")
for kp in kps[:5]:
    name = KEYPOINT_NAMES.get(kp['id'], f"kp_{kp['id']}")
    print(f"   {kp['id']:2d} {name:20s} x={kp['x']:7.1f} y={kp['y']:7.1f} conf={kp['confidence']:.2f}")
if len(kps) > 5:
    print(f"   ... y {len(kps) - 5} más")

# Mostrar resultado
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(frame)
axes[0].set_title('Frame original')
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
axes[1].set_title('Pose estimada')
axes[1].axis('off')
plt.tight_layout()
plt.show()

---
## 4. Calcular Ángulos de Sentadilla

A partir de los keypoints, calculamos métricas biomecánicas clave:

In [ ]:
# Convertir keypoints a dict y calcular ángulos
if kps:
    kps_dict = pose_extractor.keypoints_to_dict(kps)
    angles = get_squat_angles(kps_dict)
    
    print("📐 Ángulos calculados:\n")
    for key, value in angles.items():
        print(f"   {key:25s} = {value:8.2f}")
        if key in ANGLE_EXPLANATIONS:
            print(f"   {'':25s}   {ANGLE_EXPLANATIONS[key][:80]}...")
        print()
else:
    print("❌ No hay keypoints para calcular ángulos")
    print("   (Es normal si no hay una persona visible en el frame)")

---
## 5. Procesar un Video

Si tienes un video de sentadilla, procesalo acá. Busca archivos `.mp4` en el directorio `test_videos/`.

In [ ]:
# Buscar videos
video_dir = Path("test_videos")
videos = list(video_dir.glob("*.mp4")) + list(video_dir.glob("*.MP4")) + list(video_dir.glob("*.avi"))

if videos:
    print("📹 Videos encontrados:")
    for i, v in enumerate(videos):
        print(f"   [{i}] {v.name}")
else:
    print("📹 No hay videos en 'test_videos/'")
    print("   Copia un video .mp4 ahí y vuelve a ejecutar esta celda.")
    print("   O usa la sección 'Demo Sintética' más abajo.")

### Opción A: Procesar un Video Real
(Solo si hay videos disponibles)

In [ ]:
# --- DESCOMENTAR para procesar un video ---
# video_path = str(videos[0])
# print(f"Procesando: {video_path}")
# 
# frame_keypoints, output_video = pose_extractor.process_video(
#     video_path, frame_skip=FRAME_SKIP)
# print(f"Frames procesados: {len(frame_keypoints)}")
# print(f"Video anotado: {output_video}")

print("☝️  Descomenta el código de arriba cuando tengas un video.")

### Opción B: Demo Sintética

Generamos datos sintéticos para demostrar el clasificador sin necesidad de videos reales.

In [ ]:
# Generar datos sintéticos de ejemplo
np.random.seed(42)

def generate_synthetic_squat(is_good: bool, n_frames: int = 50):
    """Genera ángulos sintéticos para una sentadilla."""
    frames = []
    for i in range(n_frames):
        # Progresión: de pie (0%) → abajo (50%) → de pie (100%)
        phase = i / n_frames
        depth = np.sin(phase * np.pi)  # 0 → 1 → 0
        
        if is_good:
            knee_min = 80 + np.random.uniform(-5, 5)       # ~80° en el fondo
            back = 30 + depth * 10 + np.random.uniform(-3, 3)  # 30-40°
            sym = np.random.uniform(0, 5)                  # simétrica
            ktoe = np.random.uniform(-10, 20)              # rodilla controlada
        else:
            knee_min = 120 + np.random.uniform(-5, 5)      # no llega a paralelo
            back = 50 + depth * 20 + np.random.uniform(-5, 5)  # muy inclinado
            sym = np.random.uniform(10, 25)                # asimétrico
            ktoe = np.random.uniform(-30, 50)              # rodilla descontrolada
        
        # Ángulo de rodilla: de pie (~170°) → fondo (~knee_min) → de pie
        knee = 170 - (170 - knee_min) * depth + np.random.uniform(-3, 3)
        
        frames.append({
            'left_knee_angle': knee + np.random.uniform(-1, 1),
            'right_knee_angle': knee + (0 if is_good else sym),
            'left_hip_angle': 120 - depth * 40 + np.random.uniform(-2, 2),
            'right_hip_angle': 120 - depth * 40 + (0 if is_good else sym),
            'back_angle': back + np.random.uniform(-2, 2),
            'left_knee_toe_x': ktoe + np.random.uniform(-5, 5),
            'right_knee_toe_x': ktoe + np.random.uniform(-5, 5),
            'knee_symmetry': abs(sym + np.random.uniform(-1, 1)),
            'hip_symmetry': abs(sym * 0.8 + np.random.uniform(-1, 1)),
        })
    return frames

# Generar samples
good_frames = generate_synthetic_squat(is_good=True)
bad_frames = generate_synthetic_squat(is_good=False)

good_features = aggregate_video_features(good_frames)
bad_features = aggregate_video_features(bad_frames)

print(f"✅ Datos sintéticos generados:")
print(f"   Buena forma: {len(good_frames)} frames")
print(f"   Mala forma: {len(bad_frames)} frames")
print(f"\n   Features totales por video: {len(good_features)}")

---
## 6. Clasificador de Forma

Usamos el clasificador basado en reglas biomecánicas para evaluar la calidad de la sentadilla.

In [ ]:
# Crear clasificador
classifier = SquatFormClassifier()
print("✅ Clasificador creado")
print(f"   Umbrales: {len(classifier.thresholds)} parámetros")

In [ ]:
def show_evaluation(classifier, features, label: str):
    """Muestra la evaluación completa para un set de features."""
    criteria, overall = classifier.score_squat(features)
    pred = classifier.predict([features])[0]
    proba = classifier.predict_proba([features])[0]
    
    print(f"{'='*60}")
    print(f"📊 EVALUACIÓN: {label}")
    print(f"{'='*60}")
    print(f"   Predicción: {'✅ Buena forma' if pred == 0 else '⚠️  Necesita trabajo'}")
    print(f"   Confianza:  {proba[0]:.1%} buena / {proba[1]:.1%} mala")
    print(f"   Puntaje global: {overall:.1f}/100\n")
    
    print(f"   {'Criterio':25s} {'Puntaje':8s} {'Peso':6s}  {'Resultado'}")
    print(f"   {'─'*25} {'─'*8} {'─'*6}  {'─'*40}")
    for c in criteria:
        status = '✅' if c.passed else '⚠️'
        print(f"   {c.name:25s} {c.score:6.1f}/100 {c.weight:4.1f}x  {status} {c.detail[:60]}")
    print()
    
    feedback = classifier.get_feedback(features)
    print("💡 Feedback:")
    for tip in feedback:
        print(f"   {tip}")
    print()

# Evaluar ambos casos
show_evaluation(classifier, good_features, "BUENA FORMA (sintética)")
show_evaluation(classifier, bad_features, "MALA FORMA (sintética)")

---
## 7. Guardar el Modelo

Guardamos el clasificador como archivo `.pkl` para usarlo en la app Streamlit.

In [ ]:
model_path = Path("models/squat_form_model.pkl")
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(classifier, model_path)
print(f"✅ Modelo guardado: {model_path}")
print(f"   Tamaño: {model_path.stat().st_size / 1024:.1f} KB")

---
## 8. Cargar y Verificar

Verificamos que el modelo se carga correctamente y funciona.

In [ ]:
# Cargar modelo guardado
loaded = joblib.load(model_path)

# Probar que funciona
test_features = [good_features, bad_features]
preds = loaded.predict(test_features)
probas = loaded.predict_proba(test_features)

print("✅ Modelo cargado y verificado")
print(f"   Predicciones: {preds}")
print(f"   Probabilidades:\n")
for i, (p, pr) in enumerate(zip(preds, probas)):
    label = "Buena forma" if p == 0 else "Mala forma"
    print(f"   Sample {i}: {label} (good={pr[0]:.1%}, bad={pr[1]:.1%})")

---
## 9. Probar con Webcam (Tiempo Real)

Procesa frames en vivo desde la webcam y obtén feedback instantáneo. Presiona `q` para salir.

In [ ]:
def live_webcam_analysis(camera_id=0):
    """Análisis en vivo desde la webcam."""
    cap = cv2.VideoCapture(camera_id)
    if not cap.isOpened():
        print("❌ No se pudo abrir la cámara")
        return
    
    print("📸 Análisis en vivo — Presiona 'q' para salir")
    frame_count = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_count += 1
        
        # Procesar cada frame (sin skip para tiempo real)
        kps, annotated = pose_extractor.extract_from_frame(frame)
        
        # Calcular ángulos si hay keypoints
        if kps:
            kps_dict = pose_extractor.keypoints_to_dict(kps)
            angles = get_squat_angles(kps_dict)
            
            # Mostrar ángulos en el frame
            y_pos = 30
            for key, val in angles.items():
                if 'knee' in key and 'angle' in key:
                    cv2.putText(annotated, f"{key}: {val:.0f}",
                                (10, y_pos), cv2.FONT_HERSHEY_SIMPLEX,
                                0.5, (0, 255, 0), 1)
                    y_pos += 20
        
        cv2.imshow("Squat Form Analyzer - Presiona 'q' para salir", annotated)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    cap.release()
    cv2.destroyAllWindows()
    print(f"✅ Sesión finalizada — {frame_count} frames procesados")


# --- Ejecutar (descomentar para usar) ---
# live_webcam_analysis()

print("☝️  Descomenta la línea de arriba para activar la webcam.")
print("   Necesitás ejecutar en un entorno con display (no en terminal remota).")

---
## 10. Resumen

### Lo que logramos:
✅ Extracción de keypoints con YOLOv8-pose
✅ Cálculo de ángulos biomecánicos específicos para sentadillas
✅ Clasificador basado en reglas (no necesita datos etiquetados)
✅ Modelo guardado como `models/squat_form_model.pkl`
✅ App Streamlit lista en `app.py`

### Para correr la app:
```bash
streamlit run app.py
```

### Próximos pasos:
- Graba videos de sentadillas (buena y mala forma) y ponelos en `test_videos/`
- Usa el notebook para evaluarlos
- Ajusta los umbrales en `SquatFormClassifier` si es necesario
- Si juntás suficientes datos, entrena un XGBoost como en el notebook original